# Notebook 04 — scripted students and grounding probes

**Scope.** This notebook tests the authored patient environment and deterministic verifier with scripted student turns. It does not measure real students or validate clinical communication. Explicit intents are fixture inputs; no student action labels are trained or inferred. The Notebook 02 trained extractor remains **NOT EVALUABLE**.

**Sources.** [`AGENTS.md`](../../AGENTS.md), [`PRD.md`](../../PRD.md) §§9.1, 9.3, 9.4 and §15 (V-01, V-02, V-04), [`context.md`](../../context.md) §14, [`Notebook 02 report`](../docs/validation/action_data.json), [`Notebook 03 report`](../docs/validation/patient_environment.json), and the three authored cases in [`cases/`](../../cases/). Each trace records case path, SHA-256, version, fact IDs and their source fields. The external Patient–Doctor corpus remains excluded.

In [ ]:
import gzip, json, os, sys
from pathlib import Path
os.environ["LLM_ENABLED"] = "false"
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "PRD.md").exists())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "backend"))
from backend_kelsch.scripts.eval_grounding import evaluate
from backend_kelsch.app.grounding.evaluation import export_review_queue
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt
report, traces = evaluate()
print("Scripts", report["scripts"], "Gates", report["gates"])
print("Source example", traces["runs"][0]["case_source"])

## 1. Three scripted approaches across three cases

Each case receives novice, competent and expert authored sequences under straightforward, typical and difficult **threshold-only engineering vectors**. This produces 3 × 3 × 3 = 27 runs. The vectors are not complete S-11 presets and these profiles are not observed learners. The scripts include a request for a fabricated value, a request for an unrecorded diagnosis, a guarded topic and a repeated topic. Use `scripts(case_id)` in the module to inspect exact utterances and intents.

In [ ]:
from backend_kelsch.app.grounding.evaluation import scripts
for case_id in report["scripts"]["cases"]:
    print("\n", case_id)
    for level, turns in scripts(case_id).items():
        print(level, [(x.utterance, sorted(x.intent.topic_ids())) for x in turns])
print("Total turns", report["scripts"]["turn_count"])

## 2. Turn ledger with expected disclosure left blank

Inspect one run. Every row contains utterance, explicit intent, case version and source hash (on the run), trust events with exact spans, withheld and disclosed fact IDs, patient reply, cited source metadata, deterministic verifier status, and template fallback. `expected_disclosures` stays `None` until independent faculty review; the actual release is never copied into that field.

In [ ]:
walter = next(r for r in traces["runs"] if r["script_id"] == "walter_greene:expert:typical")
ledger = pd.DataFrame([{"turn": t["turn"], "student": t["utterance"],
    "trust": f"{t['trust_before']}→{t['trust_after']}",
    "actual": ", ".join(t["allowed_fact_ids"]),
    "expected": t["expected_disclosures"],
    "withheld": ", ".join(w["fact_id"] for w in t["withheld"]),
    "evidence": "; ".join(e["span"] for e in t["trust_events"]),
    "cited": ", ".join(t["cited_fact_ids"]),
    "fallback": t["verifier"]["fallback_used"]} for t in walter["turns"]])
display(ledger)
print("Case", walter["case_id"], walter["case_version"], walter["case_source"])
print("Sample cited fact sources:", next(t["fact_sources"] for t in walter["turns"] if t["fact_sources"]))

### Visual state trace

Blue steps are accumulated disclosed facts; the green line is trust after each turn. A withheld fact is not silently treated as an expected negative. The two axes describe the simulator state for one authored script.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
xs = [0] + [t["turn"] for t in walter["turns"]]
seen = set(); discovered = [0]
for t in walter["turns"]:
    seen.update(t["allowed_fact_ids"]); discovered.append(len(seen))
ax.step(xs, discovered, where="post", label="Disclosed fact IDs", color="#2176ae", linewidth=2)
ax.set(xlabel="Turn", ylabel="Disclosed facts", title="Walter expert script — typical threshold vector")
ay = ax.twinx()
ay.plot(xs, [0] + [t["trust_after"] for t in walter["turns"]], marker="o", color="#328362", label="Trust")
ay.set_ylabel("Trust")
ax.legend(loc="upper left"); ay.legend(loc="lower right"); fig.tight_layout(); plt.show()

## 3. Verifier challenge set

Five candidates per case include an approved exact fact, invented number, unauthorized citation, premature guarded reply, and unsupported prose. The first three blocked categories test deterministic checks. The prose candidate contains no new number and no unauthorized fact ID, so it passes deterministic checks. Its meaning is **not cleared**; it is never delivered. An optional, configured Bedrock judge can log a model ID and prompt version, but semantic accuracy still requires independent faculty labels.

In [ ]:
challenge_view = pd.DataFrame([{"case": x["case_id"], "kind": x["kind"],
    "reply": x["reply"], "cited": ",".join(x["cited_fact_ids"]),
    "blocked": x["deterministic_blocked"], "problems": "; ".join(x["deterministic_problems"]),
    "review": x["review_status"]} for x in traces["challenges"]])
display(challenge_view)
print("Challenge denominators:", report["challenge_set"]["counts"])
assert report["challenge_set"]["counts"]["mechanical_blocks"] == 9
assert all(not x["deterministic_blocked"] for x in traces["challenges"] if x["kind"] == "unsupported_prose")

## 4. Source coverage, replay and fallback accounting

The offline template uses approved `say` text from the case. The script checked each cited ID against disclosed facts and recorded case source metadata. This is **fact ID coverage**, not sentence-level semantic grounding. Every response here uses the template by design; `189/189` is an offline template fraction, not the live pipeline's fallback rate. All 27 engine replays match their original event and state fingerprints.

In [ ]:
m = report["mechanics"]
print("Counts:", m["counts"])
print("Fact-source coverage:", m["cited_fact_source_coverage"])
print("Offline template rate:", m["offline_template_rate"])
print("Live pipeline fallback rate:", m["pipeline_fallback_rate"])
assert m["counts"]["invariant_violations"] == 0
assert m["counts"]["replay_matches"] == m["counts"]["replays"]

## 5. Difficulty comparison is descriptive

The heatmap shows disclosed fact counts for each authored script and threshold vector. Pairwise differences are within a fixed script, with no independent student sample. In particular, per-fact `min_trust` overrides can make the three vectors identical for Alina and Eli. No significance claim or V-04 calibration follows from this plot.

In [ ]:
difficulty = pd.DataFrame(report["illustrative_difficulty"]["discovery"])
mat = difficulty.pivot(index=["case_id", "level"], columns="preset", values="disclosed")[["straightforward", "typical", "difficult"]]
fig, ax = plt.subplots(figsize=(8, 5))
image = ax.imshow(mat.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(3), mat.columns)
ax.set_yticks(range(len(mat)), [f"{case}: {level}" for case, level in mat.index])
for i in range(len(mat)):
    for j in range(3): ax.text(j, i, str(mat.iloc[i, j]), ha="center", va="center")
ax.set_title("Disclosures per authored script (illustrative thresholds)")
fig.colorbar(image, ax=ax, label="Fact IDs"); fig.tight_layout(); plt.show()
display(pd.DataFrame(report["illustrative_difficulty"]["paired_differences"]))

## 6. Independent review workflow and gates

Use `--export-review-queue queue.jsonl` to prepare *pending* records. A faculty reviewer supplies expected release IDs for **all 189 turns**, and supported/unsupported decisions for **all 15 challenge replies**. Each row must carry the exact case SHA-256/version, independent reviewer ID, role and timestamp. `--reviews reviewed.jsonl` checks the schema and source lock. Partial review yields null precision/recall. The template itself never fills expected fields from actual releases. A complete semantic review additionally needs a verdict for every deterministically permitted candidate before semantic metrics exist.

| Gate | PASS | FAIL | Current |
|---|---|---|---|
| Offline mechanics | Zero fact/access invariant violations; 27/27 replay; 9/9 mechanical blocks and 3/3 exact supported passes | Any miss | PASS |
| Disclosure accuracy V-02 | All 189 turns independently reviewed, no false release or miss; report TP/FP/FN and precision/recall | Evaluated mismatch | NOT EVALUABLE |
| Semantic grounding V-01 | All 15 replies reviewed and every permitted reply judged; zero unsupported permitted claims | Evaluated unsupported permit | NOT EVALUABLE |
| V-04 calibration | Faculty-approved full presets and sampled student protocols | Evaluated failure under agreed criterion | NOT EVALUABLE |
| Notebook 02 action model | Its reviewed-label split and performance gates pass | Evaluated below targets | NOT EVALUABLE |

Script commands and limits appear in the [validation report](../docs/validation/grounding_eval.md).

In [ ]:
from tempfile import TemporaryDirectory
with TemporaryDirectory() as d:
    queue = Path(d) / "review_queue.jsonl"
    export_review_queue(queue, traces["runs"], traces["challenges"])
    preview = [json.loads(line) for line in queue.read_text().splitlines()[:2]]
    display(pd.DataFrame(preview)[["kind", "script_id", "turn", "expected_fact_ids", "review_status"]])
print("Reviewed release turns:", report["reviews"]["reviewed_release_turns"], "/", report["reviews"]["total_release_turns"])
print("Reviewed semantic candidates:", report["reviews"]["reviewed_challenges"], "/", report["reviews"]["total_challenges"])
print("Disclosure precision/recall:", report["disclosure_metrics"])
print("Semantic accuracy:", report["challenge_set"]["semantic_accuracy"])
print("Gates:", report["gates"])